In [1]:
from pathlib import Path
import zipfile
import pandas as pd

# --------------------------------------------------
# Settings
# --------------------------------------------------

downloads_dir = Path.home() / "Downloads"
extract_dir = downloads_dir / "patstat_extracted_resulttables"
output_file = downloads_dir / "patstat_combined_results.csv"

extract_dir.mkdir(exist_ok=True)

# Find zipped PATSTAT result files
zip_files = sorted(downloads_dir.glob("resulttable-*.zip"))

if not zip_files:
    raise FileNotFoundError(f"No resulttable-*.zip files found in {downloads_dir}")

print(f"Found {len(zip_files)} ZIP file(s).")

dataframes = []

# --------------------------------------------------
# Helper function: read PATSTAT resulttable
# --------------------------------------------------

def read_patstat_resulttable(file_path):
    """
    Tries to read a PATSTAT resulttable file even if it has no extension.
    Tests common separators: comma, semicolon, tab.
    """

    separators = [";"]

    for sep in separators:
        try:
            df = pd.read_csv(
                file_path,
                sep=sep,
                encoding="utf-8-sig",
                engine="python"
            )

            # Accept the read only if it produced more than one column
            if df.shape[1] > 1:
                print(f"Read successfully with separator: {repr(sep)}")
                return df

        except Exception:
            pass

    # Last fallback: try default read_csv
    return pd.read_csv(file_path, encoding="utf-8-sig", engine="python")


# --------------------------------------------------
# Unzip and read each resulttable
# --------------------------------------------------

for zip_path in zip_files:
    print(f"\nProcessing ZIP: {zip_path.name}")

    # Create a separate extraction folder for each ZIP
    this_extract_dir = extract_dir / zip_path.stem
    this_extract_dir.mkdir(exist_ok=True)

    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(this_extract_dir)

    # Find files named resulttable, resulttable.csv, etc.
    result_files = [
        p for p in this_extract_dir.rglob("*")
        if p.is_file() and p.name.lower().startswith("resulttable")
    ]

    if not result_files:
        print(f"No resulttable file found inside {zip_path.name}")
        continue

    for result_file in result_files:
        print(f"Reading file inside ZIP: {result_file.name}")

        df = read_patstat_resulttable(result_file)

        # Track source
        df["source_zip"] = zip_path.name
        df["source_internal_file"] = result_file.name

        dataframes.append(df)


# --------------------------------------------------
# Combine and export
# --------------------------------------------------

if not dataframes:
    raise FileNotFoundError("No readable resulttable files were found.")

combined_df = pd.concat(dataframes, ignore_index=True)

# Optional: remove exact duplicate rows
combined_df = combined_df.drop_duplicates()

combined_df.to_csv(output_file, index=False, encoding="utf-8-sig")

print("\nDone.")
print(f"Combined rows: {len(combined_df)}")
print(f"Combined columns: {len(combined_df.columns)}")
print(f"Saved to: {output_file}")

Found 10 ZIP file(s).

Processing ZIP: resulttable-20260525005541.zip
Reading file inside ZIP: resulttable.csv
Read successfully with separator: ';'

Processing ZIP: resulttable-20260525005620.zip
Reading file inside ZIP: resulttable.csv
Read successfully with separator: ';'

Processing ZIP: resulttable-20260525005631.zip
Reading file inside ZIP: resulttable.csv
Read successfully with separator: ';'

Processing ZIP: resulttable-20260525010155.zip
Reading file inside ZIP: resulttable.csv
Read successfully with separator: ';'

Processing ZIP: resulttable-20260525010200.zip
Reading file inside ZIP: resulttable.csv
Read successfully with separator: ';'

Processing ZIP: resulttable-20260525010210.zip
Reading file inside ZIP: resulttable.csv
Read successfully with separator: ';'

Processing ZIP: resulttable-20260525010220.zip
Reading file inside ZIP: resulttable.csv
Read successfully with separator: ';'

Processing ZIP: resulttable-20260525010230.zip
Reading file inside ZIP: resulttable.csv
